In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import json
import plotly.graph_objects as go
from datetime import date, timedelta, datetime
import joblib

import utils
import network

sequence_length = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

statistics_df = pd.read_csv('../../Data_processed/Statistics.csv')

# WEATHER_MIN = pd.Timestamp("2011-11-01 00:00:00")
# WEATHER_MAX = pd.Timestamp("2014-03-31 23:00:00")

medoid_config_file = 'Config/medoid_config.json'
m2s_config_file = 'Config/m2s_config.json'

with open(medoid_config_file, 'r') as f:
    medoid_config = json.load(f)

with open(m2s_config_file, 'r') as f:
    m2s_config = json.load(f)

model_medoid = network.m_VAE(**medoid_config).to(device)
model_medoid.load_state_dict(torch.load('../../Models/medoid_vae_best.pt'))

# model_m2s = network.m2s_VAE(**m2s_config).to(device)
model_m2s = network.m2s_VAE(**m2s_config).to(device)
model_m2s.load_state_dict(torch.load('../../Models/m2s_vae_test.pt'))

cuda


<All keys matched successfully>

In [2]:
def process_weather_df(weather_df):
    weather_df['tstp'] = pd.to_datetime(weather_df['tstp'])
    
    # Min-max normalization
    for col in ['temperature', 'windSpeed', 'humidity']:
        min_col = weather_df[col].min()
        max_col = weather_df[col].max()
        weather_df[col] = (weather_df[col] - min_col) / (max_col - min_col)

    weather_df = weather_df.round(4)

    # Add half-hourly time stamps by take the average of the time stamps
    weather_df.set_index('tstp', inplace=True)
    weather_df = weather_df.resample('15T').mean().interpolate()
    weather_df = weather_df.reset_index()
    return weather_df

In [3]:
start_time = pd.Timestamp("2013-07-01 00:00:00")
end_time = pd.Timestamp("2013-07-31 23:30:00")

weather_df = utils.weather_info(start_time, end_time)
print(weather_df.head())

                 tstp  temperature  windSpeed  humidity
0 2013-07-01 00:00:00       0.1156     0.5638    0.7432
1 2013-07-01 00:30:00       0.1125     0.4938    0.7500
2 2013-07-01 01:00:00       0.1093     0.4238    0.7568
3 2013-07-01 01:30:00       0.1093     0.4181    0.7568
4 2013-07-01 02:00:00       0.1093     0.4125    0.7568


In [4]:
# interpolate weather data to 15 min intervals
weather_df = process_weather_df(weather_df)
print(weather_df.head())

                 tstp  temperature  windSpeed  humidity
0 2013-07-01 00:00:00      0.11560     0.5638    0.7432
1 2013-07-01 00:15:00      0.11405     0.5288    0.7466
2 2013-07-01 00:30:00      0.11250     0.4938    0.7500
3 2013-07-01 00:45:00      0.11090     0.4588    0.7534
4 2013-07-01 01:00:00      0.10930     0.4238    0.7568


In [5]:
def generate_synthetic_energy(start_date_time = None, end_date_time = None, statistics = None, spike_hours = None, pre_spike_length = None, post_spike_length = None, model_medoid = None, model_m2s = None, device = None):
    '''
    Generate synthetic energy data

    start_date_time: string with the start date and time in the format 'YYYY-MM-DD HH:MM:SS'
    end_date_time: string with the end date and time in the format 'YYYY-MM-DD HH:MM:SS'

    statistics: list with the statistics in the following order: mean, median, std, min, max, gradient

    spike_hours: list of strings in the format 'HH:MM:SS'
    spike_durations: list of integers
    Should be the same length

    Returns a DataFrame with the time, and the synthetic energy data

    time_df
    tstp       
    2013-06-25 00:00:00   
    2013-06-25 00:30:00
    2013-06-25 01:00:00
    ...

    synthetic_energy
    [0.1, 0.2, 0.15, ...]

    '''
    weather_df = utils.weather_info(start_date_time, end_date_time)
    weather_df = process_weather_df(weather_df)
    time_df = weather_df[['tstp']]

    if spike_hours and pre_spike_length and post_spike_length:
        spike_magnitude = statistics[4]
        spike_df = utils.get_spike_df_input(spike_hours, pre_spike_length, post_spike_length, spike_magnitude)
        w_spike_df = utils.merge_weather_spike(weather_df, spike_df)
    else:
        mean = statistics[0]
        std  = statistics[2]
        max = statistics[4]
        statistics_df = pd.DataFrame({'mean': statistics[0], 'median': statistics[1], 'std': statistics[2], 'min': statistics[3], 'max': statistics[4], 'gradient': statistics[5]}, index=[0])
        X = statistics_df.values
        kmoid_cluster = joblib.load('../Data_preprocess/kmedoids_model.joblib')
        cluster = kmoid_cluster.predict(X)[0]
        month = int(start_date_time.split('-')[1])

        spike_df = utils.get_spike_df(cluster, month, mean, std, max)
        start_date_str = start_date_time.split()[0]
        end_date_str = end_date_time.split()[0]

        # Convert the date strings to datetime objects
        start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
        end_date = datetime.strptime(end_date_str, '%Y-%m-%d')

        # Calculate the difference in days
        num_days = (end_date - start_date).days + 1  # Including both start and end dates

        spike_df_days = []
        
        for i in range(num_days):
            spike_df = utils.get_spike_df(cluster, month, mean, std, max)
            spike_df_days.append(spike_df)

        spike_df_days = pd.concat(spike_df_days, ignore_index=True)
        w_spike_df = utils.trim_and_merge_spike_weather(weather_df, spike_df_days)
        
    w_spike_df = utils.cyclic_time(w_spike_df)

    w_spike_s_df = utils.merge_statistics(w_spike_df, statistics)

    noise = torch.randn(1, len(w_spike_s_df), utils.get_m_latent_dim()).to(device)

    weather_columns = ['temperature', 'windSpeed', 'humidity']
    cluster_columns = ['kmedoid_clusters']
    time_columns = ['date_sin', 'date_cos', 'time_sin', 'time_cos']
    statistical_columns = ['mean', 'median', 'std',  'min', 'max', 'gradient']
    spike_columns = ['spike_type', 'spike_magnitude']

    weather = torch.tensor(w_spike_s_df[weather_columns].values, dtype=torch.float32).unsqueeze(0).to(device)
    cluster = torch.tensor(w_spike_s_df[cluster_columns].values, dtype=torch.float32).unsqueeze(0).to(device).int()
    time = torch.tensor(w_spike_s_df[time_columns].values, dtype=torch.float32).unsqueeze(0).to(device)
    statistical = torch.tensor(w_spike_s_df[statistical_columns].values, dtype=torch.float32).unsqueeze(0).to(device)
    spike = torch.tensor(w_spike_s_df[spike_columns].values, dtype=torch.float32).unsqueeze(0).to(device).int()

    spike_type = spike[:, :, 0:1].int().to(device)
    spike_magnitude = spike[:, :, 1:].to(device)

    h = model_medoid.lstm_decoder(noise, weather, cluster, time)
    # Add noise to the hidden state
    h = h + torch.randn(h.shape).to(device) * 0.1

    latent_space = model_m2s.lstm_encoder(h)
    mu, logvar = latent_space.chunk(2, dim=2)
    z = model_m2s.reparameterize(mu, logvar)
    h = model_m2s.lstm_decoder(z, weather, cluster, time, statistical, spike_type, spike_magnitude)

    synthetic_energy = h.squeeze().detach().cpu().numpy()

    min_energy = statistical.squeeze().detach().cpu().numpy()[0][3]
    gradient = statistical.squeeze().detach().cpu().numpy()[0][-1]
    enhanced_synthetic_energy = utils.enhanced_energy_profile(spike_type, synthetic_energy, gradient, min_energy)

    return time_df, synthetic_energy, enhanced_synthetic_energy

In [6]:
def sample_stats(statistics_df, rng: np.random.Generator):
    row = statistics_df.sample(
        1, random_state=int(rng.integers(0, 2**32 - 1))
    ).iloc[0]

    mean_v     = max(0.001, row['mean']     + rng.uniform(-0.05, 0.05))
    median_v   = max(0.001, row['median']   + rng.uniform(-0.05, 0.05))
    std_v      = max(0.001, row['std']      + rng.uniform(-0.05, 0.05))
    min_v      = max(0.001, row['min']      + rng.uniform(-0.01, 0.01))
    max_v      = max(0.001, row['max']      + rng.uniform(-0.05, 0.05))
    gradient_v = max(0.001, row['gradient'] + rng.uniform(-0.01, 0.01))

    stats = [round(mean_v, 4), round(median_v, 4), round(std_v, 4),
             round(min_v, 4), round(max_v, 4), round(gradient_v, 4)]
    return row.get("LCLid", None), stats

def bulk_generate(
    n_samples: int,
    statistics_df: pd.DataFrame,
    model_medoid,
    model_m2s,
    device,
    out_path: str,
    min_days=1,
    max_days=30,
    seed=0,
    chunk_size=200,
):
    rng = np.random.default_rng(seed)

    buffer = []
    written_rows = 0

    for idx in range(n_samples):

        lclid, stats = sample_stats(statistics_df, rng)

        # IMPORTANT: generate_synthetic_energy_t must already be defined in the notebook
        time_df, syn, syn_enh = generate_synthetic_energy(
            start_date_time=start_time.strftime("%Y-%m-%d %H:%M:%S"),
            end_date_time=end_time.strftime("%Y-%m-%d %H:%M:%S"),
            statistics=stats,
            model_medoid=model_medoid,
            model_m2s=model_m2s,
            device=device,
        )

        df = time_df.copy()
        df["energy_syn(kWh/hh)"] = syn_enh[:len(df)]

        buffer.append(df)

        # flush chunk to disk
        if len(buffer) >= chunk_size:
            chunk = pd.concat(buffer, ignore_index=True)
            mode = "w" if written_rows == 0 else "a"
            header = (written_rows == 0)
            chunk.to_csv(out_path, index=False, mode=mode, header=header)
            written_rows += len(chunk)
            buffer = []

    # final flush
    if buffer:
        chunk = pd.concat(buffer, ignore_index=True)
        mode = "w" if written_rows == 0 else "a"
        header = (written_rows == 0)
        chunk.to_csv(out_path, index=False, mode=mode, header=header)
        written_rows += len(chunk)

    return out_path


In [63]:
out_csv = bulk_generate(
    n_samples=1,
    statistics_df=statistics_df,
    model_medoid=model_medoid,
    model_m2s=model_m2s,
    device=device,
    out_path="synthetic_bulk_1.csv",
    min_days=7,
    max_days=7,       # fixed 7-day windows
    seed=3,
    chunk_size=50,    # smaller chunk = more frequent writes
)

print("Saved to:", out_csv)


Saved to: synthetic_bulk_1.csv


In [64]:
# plot the synthetic energy profile
test_df = pd.read_csv(out_csv)
fig = go.Figure()
fig.add_trace(go.Scatter(x=test_df['tstp'], y=test_df['energy_syn(kWh/hh)'], mode='lines', name='Synthetic Energy'))
fig.update_layout(title='Synthetic Energy Profile', xaxis_title='Time', yaxis_title='Energy (kWh/hh)')
fig.show()

In [8]:
num_user = 1000
for user in range(num_user):
    out_csv = bulk_generate(
        n_samples=1,
        statistics_df=statistics_df,
        model_medoid=model_medoid,
        model_m2s=model_m2s,
        device=device,
        seed=user*7,
        out_path=f"SEED/synthetic_user_{user+1}.csv",
    )

    print("Saved to:", out_csv)

Saved to: SEED/synthetic_user_1.csv
Saved to: SEED/synthetic_user_2.csv
Saved to: SEED/synthetic_user_3.csv
Saved to: SEED/synthetic_user_4.csv
Saved to: SEED/synthetic_user_5.csv
Saved to: SEED/synthetic_user_6.csv
Saved to: SEED/synthetic_user_7.csv
Saved to: SEED/synthetic_user_8.csv
Saved to: SEED/synthetic_user_9.csv
Saved to: SEED/synthetic_user_10.csv
Saved to: SEED/synthetic_user_11.csv
Saved to: SEED/synthetic_user_12.csv
Saved to: SEED/synthetic_user_13.csv
Saved to: SEED/synthetic_user_14.csv
Saved to: SEED/synthetic_user_15.csv
Saved to: SEED/synthetic_user_16.csv
Saved to: SEED/synthetic_user_17.csv
Saved to: SEED/synthetic_user_18.csv
Saved to: SEED/synthetic_user_19.csv
Saved to: SEED/synthetic_user_20.csv
Saved to: SEED/synthetic_user_21.csv
Saved to: SEED/synthetic_user_22.csv
Saved to: SEED/synthetic_user_23.csv
Saved to: SEED/synthetic_user_24.csv
Saved to: SEED/synthetic_user_25.csv
Saved to: SEED/synthetic_user_26.csv
Saved to: SEED/synthetic_user_27.csv
Saved to: 